In [833]:
import numpy as np
import pandas as pd
import pickle as pkl
import json
import re
from dotenv import load_dotenv
import os
import requests
import gc

In [834]:
FOCUS_DROP_IDENTIFIERS = [
    'daily_grind_chance',
    'daily_grind_guaranteed'
]

CHARACTER_ACTIVITIES_KEY = 'characterActivities'
DATA_KEY = 'data'
AVAILABLE_ACTIVITIES_KEY = 'availableActivities'
VISIBLE_REWARDS_KEY = 'visibleRewards'
REWARDS_ITEMS_KEY = 'rewardItems'
UISTYLE_KEY = 'uiStyle'
ITEM_QUANTITY_KEY = 'itemQuantity'
ACTIVITY_HASH_KEY = 'activityHash'

load_dotenv()

# BUNGIE
API_KEY = os.getenv("BUNGIE_API_KEY")
MEMBERSHIP_ID = os.getenv("MEMBERSHIP_ID")
MEMBERSHIP_TYPE = os.getenv("MEMBERSHIP_TYPE")

REQUEST_HEADERS = {"X-API-Key": API_KEY}
BASE = "https://www.bungie.net"
MANIFEST_URL = "/Platform/Destiny2/Manifest/"
PROFILE_URL = f"/Platform/Destiny2/{MEMBERSHIP_TYPE}/Profile/{MEMBERSHIP_ID}/"

# JSON Paths
MANIFEST_FILENAME = "Manifest.json"
ACTIVITY_DEFINITION_FILENAME = "DestinyActivityDefinition.json"
INVENTORY_ITEM_LITE_DEFINITION_FILENAME = "DestinyInventoryItemLiteDefinition.json"
CHARACTER_DEFINITION_FILENAME = "Character.json"

In [835]:
## EXCEPTIONS ##
class FocusError(Exception):
    def __init__(self, message):
        super().__init__(message)

class FocusKeyError(FocusError):
    def __init__(self, key, message='Key not found'):
        self.key = key
        super().__init__(f'{message}: given key {key} was not found.')
        
class FocusNoValuesFoundError(FocusError):
    def __init__(self, message='No values found'):
        super().__init__(f'{message}')

class FocusHashCountError(FocusError):
    def __init__(self, activity_count, item_count, message='Item and activity count not equal'):
        super().__init__(f'{message}: {activity_count} activities and {item_count} items were found')

In [836]:
def get_data_from_file(filename):
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError as e:
        print(f'FileNotFoundError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except IsADirectoryError as e:
        print(f'IsADirectoryError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except PermissionError as e:
        print(f'PermissionError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except TimeoutError as e:
        print(f'TimeoutError {e.errno} when getting data from file: {e.strerror if (e.strerror is not None) else 'No message associated with error'}')
    except UnicodeDecodeError as e:
        print(f'TimeoutError when getting data from file: {e}')


In [837]:
def get_activities_dict(activity_data):
    # get teh data for characters
    activity_found_data = activity_data.get(CHARACTER_ACTIVITIES_KEY)
    if activity_found_data is None:
        raise FocusKeyError(CHARACTER_ACTIVITIES_KEY, 'Key not found during get_activities_dict()')

    activity_found_data = activity_found_data.get(DATA_KEY)
    if activity_found_data is None:
        raise FocusKeyError(DATA_KEY, 'Key not found during get_activities_dict()')

    activity_found_data = activity_found_data.values()
    if (activity_found_data is None) or (len(activity_found_data) == 0):
        raise FocusNoValuesFoundError('No activities found during get_activities_dict()')

    # grab the data from teh first character (all characters have the same Focus Drops,
    # barring class distinctions (i.e. Warlock Bond, Titan Helm, etc)
    # therefore, we don't need a specific character ID, which could present
    # problems in the future.
    activity_found_data = next(iter(activity_found_data))
    if activity_found_data is not None:
        activity_found_data = activity_found_data.get(AVAILABLE_ACTIVITIES_KEY)
        if activity_found_data is not None:
            return activity_found_data
        else:
            raise FocusKeyError(AVAILABLE_ACTIVITIES_KEY, 'Key not found during get_activities_dict()')
    else:
        raise FocusNoValuesFoundError('No activities found during get_activities_dict()')



In [838]:
def activity_df_from_data(activity_data):
    activities_df = pd.DataFrame(activity_data)
    if activities_df.empty or activities_df is None:
        raise FocusNoValuesFoundError('Unable to build DataFrame from activity data in activity_data_to_df()')
    # explode 'visibleRewards' lists into rows
    activities_df = activities_df.explode(VISIBLE_REWARDS_KEY)
    if activities_df is None or activities_df.empty or activities_df.size == 0:
        raise FocusNoValuesFoundError('Exploded DataFrame in activity_data_to_df() is empty')
    return activities_df

In [839]:
########################################
# DataFrame Helper Functions
########################################

def extract_reward_items(value):
    value = value.iloc[0]
    if pd.notna(value) is True:
        if value is not None:
            return value.get(REWARDS_ITEMS_KEY)
        else:
            return None
    else:
        return None


def extract_uistyle_validity(value):
    value = value.get(REWARDS_ITEMS_KEY)
    if value is not None:
        uistyle = value.get(UISTYLE_KEY)
        if uistyle is not None:
            if uistyle in FOCUS_DROP_IDENTIFIERS:
                return True
            else:
                return False
        else:
            return False
    else:
        return False

def extract_item_hash(value):
    value = value.iloc[0]
    if value is None:
        return None
    value = value.get(ITEM_QUANTITY_KEY)
    if value is None:
        return None
    value = value.get('itemHash')
    if value is None:
        return None
    else:
        return value

In [840]:
def rewards_df_from_activity_df(exploded_activity_df):
    # extract series from visibleRewards
    reward_series = exploded_activity_df.get([VISIBLE_REWARDS_KEY]).apply(extract_reward_items, axis=1)

    # for entry in reward_series:
    #     display(entry)
    # with open('dump.txt', 'w') as f:
    #     json.dump(, f)
    reward_series.to_csv('dump.txt', index=False, header=False)
    if reward_series is None or reward_series.empty or reward_series.size == 0:
        raise FocusNoValuesFoundError('No rewards found during rewards_df_from_activity_df()')

    # update dataframe with new series appended and explode rewardItems
    rewards_exploded_df = exploded_activity_df.assign(rewardItems=reward_series).explode(REWARDS_ITEMS_KEY)
    return rewards_exploded_df

In [841]:
def focus_items_df_from_rewards_df(rewards_exploded_df):
    # extract uiStyle validity (is a focus drop?)
    uistyle_extracted = rewards_exploded_df.apply(extract_uistyle_validity, axis=1)
    # mask dataframe with new series
    df = rewards_exploded_df[uistyle_extracted]
    if df is None or df.empty or df.size == 0:
        raise FocusNoValuesFoundError('No items found during focus_items_df_from_rewards_df()')
    return df

In [842]:
def extract_activity_item_hashes(filtered_df):
    item_hashes = filtered_df.get([REWARDS_ITEMS_KEY]).apply(extract_item_hash, axis=1)
    if item_hashes is None or item_hashes.empty or item_hashes.size == 0:
        raise FocusNoValuesFoundError('No item hashes found during extract_activity_item_hashes()')
    activity_hashes = filtered_df.get(ACTIVITY_HASH_KEY)
    if activity_hashes is None or activity_hashes.empty or activity_hashes.size == 0:
        raise FocusNoValuesFoundError('No hashes found during extract_activity_item_hashes()')
    if item_hashes.size != activity_hashes.size:
        raise FocusHashCountError(activity_hashes.size, item_hashes.size, 'Unequal hash counts in extract_activity_item_hashes()')

    # Remove duplicate activities and merge the hashes together in a list of dictionaries
    return merge_activity_item_hashes(activity_hashes, item_hashes)


In [843]:
# Remove duplicate activities and merge the hashes together in a list of dictionaries
def merge_activity_item_hashes(activity_hashes, item_hashes):
    activity_hashes.to_csv('activity_hashes.csv')
    item_hashes.to_csv('item_hashes.csv')

    activity_hashes_list = []
    item_hashes_list = []
    for i in range(len(activity_hashes)):
        item_hash = int(item_hashes.iloc[i])
        if item_hash not in item_hashes_list:
            item_hashes_list.append(item_hash)
            activity_hashes_list.append(int(activity_hashes.iloc[i]))
    return activity_hashes_list, item_hashes_list

In [844]:
def request_manifest(args):
    """
    Requests the Bungie API manifest
    :param args: can be a list of arguments, all that matters is that
        when concatenated they form a complete API request for the
        manifest.
    :return: returns a JSON object for query.
    """
    #local_manifest = None
    #remote_manifest = None
    up_to_date = True

    request_url = ""
    for arg in args:
        request_url += arg
    try:
        response = requests.get(request_url, headers=REQUEST_HEADERS, timeout=10)
        response.raise_for_status()
        remote_manifest = response.json()

        if os.path.exists(MANIFEST_FILENAME):
            with open(MANIFEST_FILENAME, "r") as f:
                local_manifest = json.load(f)
                print("Manifest loaded from local storage")
        else:
            print("Local manifest not found. Saving API manifest")
            with open(MANIFEST_FILENAME, "w") as f:
                # noinspection PyTypeChecker
                json.dump(remote_manifest, f)
            local_manifest = remote_manifest

        # this checks the need to update the manifest
        if local_manifest.get("Response", {}).get("version") != remote_manifest.get("Response", {}).get("version"):
            print(f"Local manifest version diff from remote. Updating manifest")
            print(f"Local version: {local_manifest.get("Response", {}).get("version")}")
            print(f"Remote version: {remote_manifest.get("Response", {}).get("version")}")
            with open(MANIFEST_FILENAME, "w") as f:
                # noinspection PyTypeChecker
                json.dump(remote_manifest, f)
            local_manifest = remote_manifest
            up_to_date = False

    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching manifest metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching manifest metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse manifest metadata: {e}")

    del response
    del remote_manifest
    gc.collect()

    return local_manifest, up_to_date

In [845]:
def request_activity_hashes(bng_manifest, args):
    request_url = ""
    for arg in args:
        request_url += arg

    request_url += bng_manifest["Response"]["jsonWorldComponentContentPaths"]["en"]["DestinyActivityDefinition"]
    try:
        response = requests.get(request_url, headers=REQUEST_HEADERS, timeout=10)
        response.raise_for_status()
        hashes = response.json()
    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching activity metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching activity metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse activity metadata: {e}")
    return hashes

In [846]:
def request_item_hashes(bng_manifest, args):
    request_url = ""
    for arg in args:
        request_url += arg

    request_url += bng_manifest["Response"]["jsonWorldComponentContentPaths"]["en"]["DestinyInventoryItemDefinition"]
    try:
        response = requests.get(request_url, headers=REQUEST_HEADERS)
        response.raise_for_status()
        hashes = response.json()
    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching item metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching item metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse item metadata: {e}")
    return hashes

In [847]:
def get_profile_activities():
    params = {
        "components": 204  # CharacterActivities
    }
    try:
        response = requests.get(BASE+PROFILE_URL, headers=REQUEST_HEADERS, params=params)

        if response.status_code == 200:
            character_data = response.json()["Response"]
            with open(CHARACTER_DEFINITION_FILENAME, 'w') as f:
                json.dump(character_data, f)
        else:
            print("Error:", response.status_code, response.text)
    except requests.exceptions.Timeout:
        raise RuntimeError("⏳ Bungie API timed out while fetching profile metadata.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"❌ Bungie API returned HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"⚠️ Network error while fetching profile metadata: {e}")
    except (json.JSONDecodeError, KeyError) as e:
        raise RuntimeError(f"⚠️ Could not parse profile metadata: {e}")

In [848]:
def update_json():
    manifest, up_to_date = request_manifest([BASE, MANIFEST_URL])
    activities_hashes, items_hashes = None, None

    if up_to_date:
        # check if activity JSON exists
        if os.path.exists(ACTIVITY_DEFINITION_FILENAME):
            with open(ACTIVITY_DEFINITION_FILENAME, "r") as f:
                activities_hashes = json.load(f)
                print("Activity hashes loaded from local storage")
        else:
            activity_hashes = request_activity_hashes(manifest, BASE)
            with open(ACTIVITY_DEFINITION_FILENAME, "w") as f:
                json.dump(activity_hashes, f)
                print("Activity hash file not found. Activity hashes loaded from API")

        # check if item JSON exists
        if os.path.exists(INVENTORY_ITEM_LITE_DEFINITION_FILENAME):
            with open(INVENTORY_ITEM_LITE_DEFINITION_FILENAME, "r") as f:
                items_hashes = json.load(f)
                print("Item hashes loaded from local storage")
        else:
            item_hashes = request_item_hashes(manifest, BASE)
            with open(INVENTORY_ITEM_LITE_DEFINITION_FILENAME, "w") as f:
                json.dump(item_hashes, f)
                print("Item hash file not found. Item hashes loaded from API")

        if not os.path.exists(CHARACTER_DEFINITION_FILENAME):
            get_profile_activities()
        character_data = get_data_from_file(CHARACTER_DEFINITION_FILENAME)

        # if everything exists, item and activity hashes were loaded from local memory

    else:
        activities_hashes = request_activity_hashes(manifest, BASE)
        with open(ACTIVITY_DEFINITION_FILENAME, "w") as f:
            json.dump(activities_hashes, f)

        items_hashes = request_item_hashes(manifest, BASE)
        with open(INVENTORY_ITEM_LITE_DEFINITION_FILENAME, "w") as f:
            json.dump(items_hashes, f)

        get_profile_activities()
        character_data = get_data_from_file(CHARACTER_DEFINITION_FILENAME)

    return activities_hashes, items_hashes, character_data

In [849]:
def program_start():

    # TODO: LOAD MANIFEST -> CHECK VERSION -> UP-TO-DATE ? PULL DATA FROM .TXT FILE : REQUEST & PARSE & SAVE TO .TXT FILE
    # TODO: NO NEED TO LOAD THE JSON AND PARSE EVERY SINGLE TIME, JUST DO REQUEST & PARSE ONCE A DAY

    activity_data, item_data, character_data = None, None, None
    activity_data, item_data, character_data = update_json()

    character_data = get_data_from_file('Character.json')
    if character_data is None or activity_data is None or item_data is None:
        raise Exception('No character_data')

    try:
        character_data = get_activities_dict(character_data)
        character_data = activity_df_from_data(character_data)
        character_data = rewards_df_from_activity_df(character_data)
        character_data = focus_items_df_from_rewards_df(character_data)
        activity_hashes, item_hashes = extract_activity_item_hashes(character_data)
        display('activity_hashes', activity_hashes)
        display('item_hashes', item_hashes)

    except FocusKeyError or FocusNoValuesFoundError or FocusHashCountError as e:
        print(f'Error parsing data: {e}')


In [850]:
program_start()

Manifest loaded from local storage
Activity hashes loaded from local storage
Item hashes loaded from local storage


'activity_hashes'

[4192328901,
 2489241976,
 3437865231,
 679760629,
 3534847306,
 210441999,
 4225173883,
 3985011110,
 775989808,
 1768099736,
 3584571989,
 2723561970,
 122763493]

'item_hashes'

[1402874079,
 3462703357,
 2676446840,
 2370945771,
 2875763009,
 1763361847,
 3554497829,
 3031404418,
 2639123099,
 260532765,
 1242785638,
 2297554989,
 1018012078]